In [6]:
import numpy as np

def B_from_nuF(
    nu_brk_GHz: float,
    F_brk_Jy: float,
    D_Mpc: float,
    eps_e: float,
    eps_b: float,
) -> float:
    """
    Magnetic field B (Gauss) from Eq. (18), with f=0.5 assumed.

    Parameters
    ----------
    nu_brk_GHz : float
        Break frequency in GHz.
    F_brk_Jy : float
        Break flux density in Jy.
    D_Mpc : float
        Distance in Mpc.
    eps_e : float
        Electron energy fraction ε_e (dimensionless).
    eps_b : float
        Magnetic energy fraction ε_b (dimensionless).


    Returns
    -------
    B_G : float
        Magnetic field in Gauss.
    """
    nu_ratio = nu_brk_GHz / 5.0
    bracket = (1.0 / eps_B) * (D_Mpc) * (eps_e) * (F_brk_Jy ** 0.5)
    B_G = 0.5761 * nu_ratio * (bracket ** (-4.0 / 19.0))
    return B_G


def R_from_nuF(
    nu_brk_GHz: float,
    F_brk_Jy: float,
    D_Mpc: float,
    eps_e: float,
    eps_B: float,
) -> float:
    """
    Radius R (cm) from Eq. (19), with f=0.5 assumed.

    Parameters
    ----------
    nu_brk_GHz : float
        Break frequency in GHz.
    F_brk_Jy : float
        Break flux density in Jy.
    D_Mpc : float
        Distance in Mpc.
    eps_e : float
        Electron energy fraction ε_e (dimensionless).
    eps_B : float
        Magnetic energy fraction ε_B (dimensionless).

    Returns
    -------
    R_cm : float
        Radius in cm.
    """
    nu_ratio = nu_brk_GHz / 5.0
    bracket = (D_Mpc ** 18.0) * (eps_B) * (F_brk_Jy ** 9.0) * (eps_e ** (-1.0))
    R_cm = 8.759e15 * (nu_ratio ** (-1.0)) * (bracket ** (1.0 / 19.0))
    return R_cm


def B_and_sigmaB(
    nu_brk_GHz: float,
    sigma_nu_GHz: float,
    F_brk_Jy: float,
    sigma_F_Jy: float,
    D_Mpc: float,
    eps_e: float,
    eps_B: float,
) -> tuple[float, float]:
    """
    Returns (B, sigma_B) with analytic propagation from nu_brk and F_brk only.

    For Eq. (18) with f=0.5:
        B ∝ nu^(+1) * F^(-2/19)
    so:
        (σB/B)^2 = (σnu/nu)^2 + ( (2/19) σF/F )^2
    """
    B = B_from_nuF(nu_brk_GHz, F_brk_Jy, D_Mpc, eps_e, eps_B)

    if nu_brk_GHz <= 0 or F_brk_Jy <= 0:
        raise ValueError("nu_brk_GHz and F_brk_Jy must be > 0 for fractional errors.")

    frac = np.sqrt((sigma_nu_GHz / nu_brk_GHz) ** 2 + ((2.0 / 19.0) * (sigma_F_Jy / F_brk_Jy)) ** 2)
    return B, B * frac


def R_and_sigmaR(
    nu_brk_GHz: float,
    sigma_nu_GHz: float,
    F_brk_Jy: float,
    sigma_F_Jy: float,
    D_Mpc: float,
    eps_e: float,
    eps_B: float,
) -> tuple[float, float]:
    """
    Returns (R, sigma_R) with analytic propagation from nu_brk and F_brk only.

    For Eq. (19) with f=0.5:
        R ∝ nu^(-1) * F^(9/19)
    so:
        (σR/R)^2 = (σnu/nu)^2 + ( (9/19) σF/F )^2
    """
    R = R_from_nuF(nu_brk_GHz, F_brk_Jy, D_Mpc, eps_e, eps_B)

    if nu_brk_GHz <= 0 or F_brk_Jy <= 0:
        raise ValueError("nu_brk_GHz and F_brk_Jy must be > 0 for fractional errors.")

    frac = np.sqrt((sigma_nu_GHz / nu_brk_GHz) ** 2 + ((9.0 / 19.0) * (sigma_F_Jy / F_brk_Jy)) ** 2)
    return R, R * frac


def mdot_over_vw(
    nu_brk_GHz: float,
    F_brk_Jy: float,
    t_days: float,
    D_Mpc: float,
    eps_e: float,
    eps_b: float,
) -> float:
    """
    Computes (Mdot / v_w) from the shown equation, with f=0.5 assumed.
    Output units follow the paper's normalization:
        (Mdot / v_w) * (1000 km/s) / (1e-4 Msun/yr)
    i.e. a dimensionless scaled quantity (as written).

    Only nu_brk and F_brk are treated as variables here; others are inputs.
    """
    if nu_brk_GHz <= 0 or F_brk_Jy <= 0 or t_days <= 0 or D_Mpc <= 0:
        raise ValueError("nu_brk_GHz, F_brk_Jy, t_days, D_Mpc must be > 0.")

    nu_ratio = nu_brk_GHz / 5.0
    t_ratio = t_days / 1.0  # days

    bracket = (eps_b**2.0) * (D_Mpc**(-2.0)) * (F_brk_Jy**(-1.0)) * (eps_e**(-2.0))
    X = 2.5e-5 * (nu_ratio**2.0) * (t_ratio**2.0) * (eps_B**(-1.0)) * (bracket ** (-24.0/19.0))
    return X


def mdot_over_vw_and_sigma(
    nu_brk_GHz: float,
    sigma_nu_GHz: float,
    F_brk_Jy: float,
    sigma_F_Jy: float,
    t_days: float,
    D_Mpc: float,
    eps_e: float,
    eps_B: float,
) -> tuple[float, float]:
    """
    Returns (X, sigma_X) for X = (Mdot / v_w) (scaled as in the equation),
    propagating uncertainties from nu_brk and F_brk only (assumed independent).

    Scaling:
        X ∝ nu^(+2) * F^(+24/19)
    so:
        (σX/X)^2 = (2 σnu/nu)^2 + ((24/19) σF/F)^2
    """
    X = mdot_over_vw(nu_brk_GHz, F_brk_Jy, t_days, D_Mpc, eps_e, eps_B)

    if nu_brk_GHz <= 0 or F_brk_Jy <= 0:
        raise ValueError("nu_brk_GHz and F_brk_Jy must be > 0 for fractional errors.")

    frac = np.sqrt((2.0 * sigma_nu_GHz / nu_brk_GHz)**2 +
                   ((24.0/19.0) * sigma_F_Jy / F_brk_Jy)**2)
    return X, X * frac





# --- example usage ---
if __name__ == "__main__":
    nu, s_nu = 1.43248, 0.265  # GHz
    F,  s_F  = 0.00159628, 0.000109  # Jy
    D = 17                 # Mpc
    eps_e, eps_B = 0.1, 0.01
    t = 33.0   #days

    B, sB = B_and_sigmaB(nu, s_nu, F, s_F, D, eps_e, eps_B)
    R, sR = R_and_sigmaR(nu, s_nu, F, s_F, D, eps_e, eps_B)
    X, sX = mdot_over_vw_and_sigma(nu, s_nu, F, s_F, t, D, eps_e, eps_B)
    print(f"B = {B:.4g} ± {sB:.2g} G")
    print(f"R = {R:.4g} ± {sR:.2g} cm")
    print(f"(Mdot/vw)scaled = {X:.4g} ± {sX:.2g}")
    row_csm = X/(4*np.pi*R**2)
    print(f"Density: {row_csm:.4g}")


B = 0.1103 ± 0.02 G
R = 1.877e+16 ± 3.5e+15 cm
(Mdot/vw)scaled = 28.26 ± 11
Density: 6.38e-33


In [20]:
# with all errors

import numpy as np

def _frac_err(x, sx):
    if x <= 0:
        raise ValueError("All variables must be > 0 for fractional error propagation.")
    return sx / x


# ---------- B (Eq 18) ----------
def B_value(nu_GHz, F_mJy, D_Mpc, eps_e, eps_B):
    F_Jy = F_mJy / 1000.0
    nu_ratio = nu_GHz / 5.0
    bracket = (1.0 / eps_B) * (D_Mpc) * (eps_e) * (F_Jy ** 0.5)
    return 0.5761 * nu_ratio * (bracket ** (-4.0 / 19.0))  # Gauss

def B_and_sigma(nu_GHz, s_nu_GHz, F_mJy, s_F_mJy, D_Mpc, s_D_Mpc, eps_e, eps_B):
    B = B_value(nu_GHz, F_mJy, D_Mpc, eps_e, eps_B)
    fe = np.sqrt(
        (1.0 * _frac_err(nu_GHz, s_nu_GHz))**2 +
        ((2.0/19.0) * _frac_err(F_mJy, s_F_mJy))**2 +
        ((4.0/19.0) * _frac_err(D_Mpc, s_D_Mpc))**2
    )
    return B, B * fe


# ---------- R (Eq 19) ----------
def R_value(nu_GHz, F_mJy, D_Mpc, eps_e, eps_B):
    F_Jy = F_mJy / 1000.0
    nu_ratio = nu_GHz / 5.0
    bracket = (D_Mpc**18.0) * (eps_B) * (F_Jy**9.0) * (eps_e**(-1.0))
    return 8.759e15 * (nu_ratio ** (-1.0)) * (bracket ** (1.0/19.0))  # cm

def R_and_sigma(nu_GHz, s_nu_GHz, F_mJy, s_F_mJy, D_Mpc, s_D_Mpc, eps_e, eps_B):
    R = R_value(nu_GHz, F_mJy, D_Mpc, eps_e, eps_B)
    fe = np.sqrt(
        (1.0 * _frac_err(nu_GHz, s_nu_GHz))**2 +
        ((9.0/19.0) * _frac_err(F_mJy, s_F_mJy))**2 +
        ((18.0/19.0) * _frac_err(D_Mpc, s_D_Mpc))**2
    )
    return R, R * fe


# ---------- (Mdot/vw) scaled expression ----------
def mdot_over_vw_value(nu_GHz, F_mJy, t_days, D_Mpc, eps_e, eps_B):
    F_Jy = F_mJy / 1000.0
    nu_ratio = nu_GHz / 5.0
    bracket = (eps_B**2.0) * (D_Mpc**(-2.0)) * (F_Jy**(-1.0)) * (eps_e**(-2.0))
    return 2.5e-5 * (nu_ratio**2.0) * (t_days**2.0) * (eps_B**(-1.0)) * (bracket ** (4/19))

def mdot_over_vw_and_sigma(nu_GHz, s_nu_GHz, F_mJy, s_F_mJy, t_days, s_t_days, D_Mpc, s_D_Mpc, eps_e, eps_B):
    X = mdot_over_vw_value(nu_GHz, F_mJy, t_days, D_Mpc, eps_e, eps_B)
    fe = np.sqrt(
        (2.0 * _frac_err(nu_GHz, s_nu_GHz))**2 +
        ((4/19.0) * _frac_err(F_mJy, s_F_mJy))**2 +
        (2.0 * _frac_err(t_days, s_t_days))**2 +
        ((8/19.0) * _frac_err(D_Mpc, s_D_Mpc))**2
    )
    return X, X * fe


epsE = [0.1, 0.33] #literature, equipartition
epsB = [0.01, 0.33] 

F_brk = 1.54133 #mJy
F_brk_e = 0.0678

nu_brk = 1.6068 #GHz
nu_brk_e = 0.243 

#distance in mag 31.4 pm 0.4 - convert using modulus formula


D = 19.1 #Mpc
D_e = 3.5

T = 57909.8 - 57896.8 #Days since explosion
T_e = 0.3

 

#literature

B, B_e = B_and_sigma(nu_brk, nu_brk_e, F_brk, F_brk_e, D, D_e, epsE[0], epsB[0])

R, R_e = R_and_sigma(nu_brk, nu_brk_e, F_brk, F_brk_e, D, D_e, epsE[0], epsB[0])

M_over_vw, M_over_vw_e = mdot_over_vw_and_sigma(nu_brk, nu_brk_e, F_brk, F_brk_e, T, T_e, D, D_e, epsE[0], epsB[0])

p_M_over_vw, p_M_over_vw_e =  K*M_over_vw, K*M_over_vw_e #physical conversion

density = (p_M_over_vw)/(4*np.pi*R**2)
density_e = (p_M_over_vw/(4*np.pi*R**2))*np.sqrt((p_M_over_vw_e/p_M_over_vw)**2 + 4*((R_e/R)**2))

#equipartition

B_ep, B_ep_e = B_and_sigma(nu_brk, nu_brk_e, F_brk, F_brk_e, D, D_e, epsE[1], epsB[1])

R_ep, R_ep_e = R_and_sigma(nu_brk, nu_brk_e, F_brk, F_brk_e, D, D_e, epsE[1], epsB[1])

M_over_vw_ep, M_over_vw_ep_e = mdot_over_vw_and_sigma(nu_brk, nu_brk_e, F_brk, F_brk_e, T, T_e, D, D_e, epsE[1], epsB[1])

p_M_over_vw_ep, p_M_over_vw_ep_e =  K*M_over_vw_ep, K*M_over_vw_ep_e #physical conversion

density_ep = (p_M_over_vw_ep)/(4*np.pi*R_ep**2)
density_ep_e = ((p_M_over_vw_ep)/(4*np.pi*R_ep**2))*np.sqrt((p_M_over_vw_ep_e/p_M_over_vw_ep)**2 + 4*((R_ep_e/R_ep)**2))

print(f"Literature values: B = {B:.4g} ± {B_e:.4g} G, R = {R:.4g} ± {R_e:.4g} cm, M_dot_over_vw = {p_M_over_vw:.4g} ± {p_M_over_vw_e:.4g} g/cm, density = {density:.4g} ± {density_e:.4g} g/cm^3")
      
print(f"Equipartition values: B = {B_ep:.4g}  ± {B_ep_e:.4g} G, R = {R_ep:.4g} ± {R_ep_e:.4g} cm, M_dot_over_vw = {p_M_over_vw_ep:.4g} ± {p_M_over_vw_ep_e:.4g} g/cm, density = {density_ep:.4g} ± {density_ep_e:.4g} g/cm^3")

Literature values: B = 0.1211 ± 0.01892 G, R = 1.838e+16 ± 4.249e+15 cm, M_dot_over_vw = 1.177e+12 ± 3.715e+11 g/cm, density = 2.772e-22 ± 1.552e-22 g/cm^3
Equipartition values: B = 0.1967  ± 0.03071 G, R = 2.075e+16 ± 4.797e+15 cm, M_dot_over_vw = 9.403e+10 ± 2.968e+10 g/cm, density = 1.738e-23 ± 9.729e-24 g/cm^3
